# Binary Decisions: Fixed Costs and Plant Sizes

In all notebooks so far, FINE could build any amount of capacity, however small, and its cost was proportional to the capacity. Real projects are often different: a pipeline needs planning and a route no matter how large it is, and power plants come in units of a certain size. FINE models such decisions with **integer variables**:

| Topic | Parameters |
|---|---|
| Costs that arise if a component is built at all | `hasIsBuiltBinaryVariable`, `bigM`, `investIfBuilt`, `opexIfBuilt` |
| Capacity in whole units | `capacityVariableDomain="discrete"`, `capacityPerPlantUnit` |

Integer variables turn the linear optimization problem (LP) into a **mixed-integer** problem (MILP), which is much harder to solve. Use them only where they matter.

## 1. The model: hydrogen in both regions

We use the model of the [Conversions notebook](13_conversions.ipynb): the industry in both regions needs hydrogen from 2030 on, which can be imported or produced by electrolyzers. The function `addHydrogenSupply` from `tutorialHelpers.py` adds these components exactly as in that notebook.

New is a **hydrogen pipeline** between the regions, a transmission of the commodity `hydrogen`. Its investment is proportional to its capacity and length. We first optimize the model with this pipeline as a normal, linear component:

In [1]:
import warnings

import pandas as pd

import fine as fn
from tutorialHelpers import addHydrogenSupply, buildBasicTutorialModel, readData

warnings.filterwarnings("ignore")  # keeps the output of this tutorial readable

distances = readData("distances.csv")
pipelineParameters = dict(
    name="Hydrogen pipeline",
    commodity="hydrogen",
    hasCapacityVariable=True,
    distances=distances,
    investPerCapacity=0.1,  # million Euro per GW and km
    opexPerCapacity=0.001,  # million Euro per GW, km and year
    interestRate=0.08,
    economicLifetime=40,
)

In [2]:
esM = buildBasicTutorialModel(commodityUnits={"hydrogen": "GW_H2"})
addHydrogenSupply(esM)
esM.add(fn.Transmission(esM=esM, **pipelineParameters))

esM.aggregateTemporally(n_clusters=8, segments=None)
esM.optimize(timeSeriesAggregation=True)

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter QCPDual to value 1


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26300.2))


CPU model: Intel(R) Core(TM) Ultra 7 255U, instruction set [SSE2|AVX|AVX2]


... (77 outputs left out)

Solved with barrier


    3491    6.3771246e+05   0.000000e+00   0.000000e+00      1s


Solved in 3491 iterations and 0.76 seconds (0.89 work units)


Optimal objective  6.377124616e+05


In [3]:
def pipelineAndElectrolyzers(esM):
    """Capacities of the pipeline (north to south) and the electrolyzers per region."""
    result = {}
    for year in esM.investmentPeriodNames:
        transmission = esM.getOptimizationSummary(
            "TransmissionModel", outputLevel=1, ip=year
        )
        conversion = esM.getOptimizationSummary(
            "ConversionModel", outputLevel=1, ip=year
        )
        result[year] = {
            "Pipeline [GW_H2]": transmission.loc[
                ("Hydrogen pipeline", "capacity", "[GW_H2]", "north"), "south"
            ],
            "Electrolyzers north [GW_el]": conversion.loc[
                ("Electrolyzers", "capacity", "[GW_el]"), "north"
            ],
            "Electrolyzers south [GW_el]": conversion.loc[
                ("Electrolyzers", "capacity", "[GW_el]"), "south"
            ],
        }
    return pd.DataFrame(result).astype(float)


linear = pipelineAndElectrolyzers(esM)
costLinear = esM.objectiveValue
linear.round(1)

,2020,2030,2040
Pipeline [GW_H2],0.0,3.1,3.1
Electrolyzers north [GW_el],0.0,11.3,18.2
Electrolyzers south [GW_el],0.0,0.0,4.5


Without a fixed cost, the model builds a small pipeline of 3.1 GW in 2030. The electrolyzers are built in the windy north, in 2030 only there.

## 2. A fixed cost with `hasIsBuiltBinaryVariable`

Now the pipeline gets a cost that arises only if it is built, independent of its capacity. Three parameters are needed:

- **`hasIsBuiltBinaryVariable=True`** adds a binary variable (0 or 1) per region (for transmissions: per connection) and investment period: 1 if new capacity is built there in that period, 0 otherwise.
- **`investIfBuilt`** is an investment that is paid whenever the binary variable is 1, in the same way as `investPerCapacity` (spread into annuities over the economic lifetime). For transmissions, it is given per km. (`opexIfBuilt` is the same for a yearly fixed cost.) We assume 5 million Euro per km, i.e. about 1.8 billion Euro for the 357 km.
- **`bigM`** is an upper limit for the capacity that is built if the binary variable is 1. It should be clearly larger than any capacity the model will build, but not much larger, since very large values make the problem harder to solve. Here, 100 GW. If the component has a `capacityMax`, it is used instead of `bigM`.

Note that the binary variable belongs to an investment period: if new capacity is built in two periods, the fixed cost is paid twice.

In [4]:
esM = buildBasicTutorialModel(commodityUnits={"hydrogen": "GW_H2"})
addHydrogenSupply(esM)
esM.add(
    fn.Transmission(
        esM=esM,
        **pipelineParameters,
        hasIsBuiltBinaryVariable=True,
        investIfBuilt=5,  # million Euro per km
        bigM=100,  # GW_H2
    )
)

esM.aggregateTemporally(n_clusters=8, segments=None)
esM.optimize(timeSeriesAggregation=True)

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter QCPDual to value 1


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26300.2))


CPU model: Intel(R) Core(TM) Ultra 7 255U, instruction set [SSE2|AVX|AVX2]


... (26 outputs left out)

Thread count was 3 (of 14 available processors)


Solution count 2: 638141 638448 


Optimal solution found (tolerance 1.00e-04)


Best objective 6.381406186167e+05, best bound 6.381406186167e+05, gap 0.0000%


In [5]:
withFixedCost = pipelineAndElectrolyzers(esM)
pd.concat(
    {"Linear": linear, "With fixed cost": withFixedCost}, names=["Pipeline"]
).round(1)

2020  2030  2040
Pipeline                                                     
Linear          Pipeline [GW_H2]              0.0   3.1   3.1
                Electrolyzers north [GW_el]   0.0  11.3  18.2
                Electrolyzers south [GW_el]   0.0   0.0   4.5
With fixed cost Pipeline [GW_H2]              0.0   0.0   0.0
                Electrolyzers north [GW_el]   0.0   6.9  13.8
                Electrolyzers south [GW_el]   0.0   4.5   8.9

With a fixed cost of about 1.8 billion Euro, a pipeline of 3 GW is no longer worth it: the model builds none. Instead, it builds electrolyzers in the south as well (4.5 GW in 2030 and 8.9 GW in 2040) and fewer in the north. A linear model cannot show this trade-off: there, a small pipeline costs as much per GW as a large one.

## 3. Whole plants with `capacityVariableDomain="discrete"`

Power plants are built in units of a certain size. With **`capacityVariableDomain="discrete"`**, the capacity of a component is a whole number of units of the size **`capacityPerPlantUnit`**. We give the gas power plants of the basic tutorial units of 0.8 GW (without the hydrogen components):

In [6]:
esM = buildBasicTutorialModel()
esM.updateComponent(
    "Gas power plants",
    {"capacityVariableDomain": "discrete", "capacityPerPlantUnit": 0.8},  # GW
)
esM.aggregateTemporally(n_clusters=8, segments=None)
esM.optimize(timeSeriesAggregation=True)

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter QCPDual to value 1


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26300.2))


CPU model: Intel(R) Core(TM) Ultra 7 255U, instruction set [SSE2|AVX|AVX2]


... (29 outputs left out)

Thread count was 3 (of 14 available processors)


Solution count 3: 597401 597529 598268 


Optimal solution found (tolerance 1.00e-04)


Best objective 5.974011254109e+05, best bound 5.974011254109e+05, gap 0.0000%


In [7]:
summary = esM.getOptimizationSummary("ConversionModel", outputLevel=1, ip=2040)
gasCapacity = summary.loc[("Gas power plants", "capacity")].iloc[0].astype(float)
pd.DataFrame(
    {"Capacity [GW]": gasCapacity, "Number of plants": gasCapacity / 0.8}
).round(2)

,Capacity [GW],Number of plants
north,32.8,41.0
south,30.4,38.0


In the basic tutorial, the gas power plants had 32.6 GW in the north and 30.4 GW in the south. Now they consist of 41 and 38 plants of 0.8 GW, i.e. 32.8 and 30.4 GW. The units are small compared with the system, so the result hardly changes.

## 4. Solving problems with integer variables

The solver handles integer variables by solving many linear problems (branch and bound). This can take much longer than a single linear problem, and the solver usually stops when its solution is proven to be within a small gap of the optimum (for Gurobi, by default 0.01 %). A few tips:

- Keep the number of integer variables small: use them only for the components where the decision really matters.
- Choose `bigM` as small as possible.
- Accept a larger gap via `optimizationSpecs`, e.g. `"MIPGap=0.01"` for 1 % with Gurobi or `"mip_rel_gap=0.01"` with HiGHS (see the [Solver Settings notebook](32_solver_settings_and_saving.ipynb)).
- `esM.optimize(..., relaxIsBuiltBinary=True)` replaces the binary variables of `hasIsBuiltBinaryVariable` by continuous ones between 0 and 1. The result is a lower bound for the cost of the problem with binary variables and shows quickly whether the fixed costs matter at all.

## Things to try

- Lower `investIfBuilt` of the pipeline to 1 million Euro per km. Is the pipeline built now?
- Give the pipeline a `capacityMin` of 10 GW. With a binary variable, it only applies if the pipeline is built.
- Give the electrolyzers units of 1 GW with `capacityVariableDomain="discrete"`.

The next notebook, [Power Plant Flexibility](44_power_plant_flexibility.ipynb), uses integer variables for the operation of power plants.